# AVIRSO — essai : votre tête qui tourne, à partir de trois vraies photos

Ce carnet part de **vos photos** (face, un peu tourné à gauche, un peu tourné à droite) et fabrique une tête qui tourne.

- Le modèle LivePortrait (code et poids sous licence MIT) fait tourner chaque photo de quelques degrés seulement.
- Le grain des cheveux, de la barbe et de la peau est **repris de vos vraies photos**, pas inventé.
- La tête est détourée sur fond sombre.

**Comment faire**

1. Menu *Exécution* → *Modifier le type d'exécution* → **GPU T4**.
2. Menu *Exécution* → **Tout exécuter**.
3. À la deuxième cellule, cliquez sur **Choisir des fichiers** et sélectionnez vos trois photos d'un coup.
4. L'animation s'affiche tout en bas, au bout d'environ trois minutes.

**Les photos** : même endroit, même lumière, bras tendu, téléphone à hauteur des yeux, visage neutre. Une de face, une avec la tête tournée d'environ 20 à 30° à gauche, une pareil à droite. Pas de profil complet.

Vos photos sont envoyées sur les serveurs de Google (Colab) le temps de l'essai, et nulle part ailleurs. Les images produites sont **générées par un modèle** : ce ne sont pas des photos.

In [ ]:
# 1. Installation : le modèle et ses poids (environ 540 Mo)
import os
%cd /content
if not os.path.exists('/content/LivePortrait/src'):
    !git clone -q --depth 1 https://github.com/KwaiVGI/LivePortrait
%cd /content/LivePortrait
!mkdir -p pw sortie
B = 'https://huggingface.co/KwaiVGI/LivePortrait/resolve/main/liveportrait'
for f in ['base_models/appearance_feature_extractor.pth', 'base_models/motion_extractor.pth', 'base_models/spade_generator.pth', 'base_models/warping_module.pth', 'retargeting_models/stitching_retargeting_module.pth']:
    !wget -q -nc -P pw {B}/{f}
!wget -q -nc -P pw https://storage.googleapis.com/mediapipe-models/image_segmenter/selfie_multiclass_256x256/float32/latest/selfie_multiclass_256x256.tflite
!pip -q install pillow-heif
!wget -q -O tete.py https://raw.githubusercontent.com/Copsenova/avirso-test/e1005da20b63cf1eddce3d57d0fa1bc6d1fddddd/essai-generation/tete.py
!ls pw | wc -l

In [ ]:
# 2. Vos photos : choisissez-les toutes d'un coup (JPEG ou HEIC)
from google.colab import files
import os, shutil, shlex
shutil.rmtree('photos', ignore_errors=True); os.makedirs('photos')
envoi = files.upload()
PHOTOS = []
for nom, contenu in envoi.items():
    chemin = os.path.join('photos', nom)
    open(chemin, 'wb').write(contenu)
    if os.path.exists(nom):
        os.remove(nom)
    PHOTOS.append(chemin)
ARGS = ' '.join(shlex.quote(p) for p in PHOTOS)
print(len(PHOTOS), 'photo(s) :', PHOTOS)

In [ ]:
# 3. Fabrication des images (une tous les 1,5 degré)
!rm -f sortie/t*
!python tete.py sortie/t {ARGS} 2>&1 | grep -v -i -E "warn|I0000|rebuild TensorFlow|LiteRT|migration guide|for details|XNNPACK|^[[:space:]]*$"

In [ ]:
# 4. Résultat : l'animation, puis cinq angles côte à côte
import base64, os
from IPython.display import HTML, Image as Montrer, display
assert os.path.exists('sortie/t.mp4'), "La fabrication (cellule 3) a échoué : regardez son message."
v = base64.b64encode(open('sortie/t.mp4', 'rb').read()).decode()
display(HTML(f'<video src="data:video/mp4;base64,{v}" autoplay loop muted playsinline controls style="width:min(100%,560px);border-radius:16px"></video>'))
display(Montrer(filename='sortie/t_planche.jpg'))
print('Images générées par un modèle, avec le grain repris de vos photos.')

In [ ]:
# 5. (facultatif) Pour garder l'animation : retirez le # de la ligne suivante et exécutez cette cellule
# files.download('sortie/t.mp4')